In [3]:
import pandas as pd 
import numpy as np 
import datetime as dt 
import itertools
import plotly as py
import plotly.graph_objs as go
import plotly.figure_factory as ff
import plotly.express as px
from dash import Dash, dcc, html, Input, Output
from scipy.stats import norm
from sklearn.preprocessing import StandardScaler

# Исследование вероятности выполнения заявок логистической компанией

Логистическая компания занимается оказанием услуг по железнодорожной перевозке.  
Ежедневно в компанию поступают заявки на транспортировку товаров. Для каждой заявки известно направление перевозки (между какими станциями нужно осуществить транспортировку товаров), объем товаров, требующих перевозку, планируемая дата отправки, учетное направление перевозки (некоторые особенности перевозки). В течение жизненного цикла заявки (от ее регистрации в системе до её закрытия) с ней могут происходить различные изменения: изменение станций отправки и доставки, изменение величины прибыли, получаемой компанией в результате осуществления услуги по доставке, отмена заявки и т.д. 
В конце месяца из существующих заявок формируется заявочная компания, которая передается в отдел оптимизации, который составляет расписание движения составов, которое старается выполнить весь объем поступивших заявок.

**Глобальная задача:** используя инструменты машинного обучения предсказать вероятность выполнения заявки в заявочной компании. 

**Задача на сегодня**: используя инструменты визуализации данных и статистического анализа выявить признаки, которые оказывают влияние выполнение заявки.

Вам дана информацию о том, как формировалась заявочная компания с 01.01.2023 по 30.09.2023. Для каждой заявки представлены все изменения, вносимые оператором и результат выполнения заявки (выполнение, перенос, отмена).

**Описание исходных данных:**
 -   `Идентификатор погрузки` - уникальный номер погрузки в системе данных компании;
 -   `Дата редактирования` - когда вносились изменения в информацию о заявке на выполнение погрузки;   
 -   `Версия погрузки` - номер редакции;
 -   `Оператор` -  человек, осуществляющий редактирование заявки;
 -   `Дата отправления` - планируемая дата отправки погрузки;        
 -   `Дата отправления факт` - фактическая дата отправки погрузки;
 -   `Учетное направление` - тип перевозки; 
 -   `Кластер назначения`- из какого региона двигается состав;     
 -   `Кластер отправления` -  в какой регион двигается состав; 
 -   `Код назначения` - станция откуда нужно перевести товары;           
 -   `Код отправления` - станция, куда нужно перевезти товары;         
 -   `Количество вагонов` - количество вагонов в составе;      
 -   `Контрибуция` - прибыль, которую получит компания за перевозку;                         
 -   `Идентификатор переноса` - была ли перенесена погрука (-1 – заявка отменена, 0 – заявка выполнена, 1 – заявка перенесена);
 -   `Длительность жизни` -  время, которое прошло между первой и последней редакцией заявки;
 -   `Время до отправки`  - время между последней редакцией перед отправкой и плановой датой отправки;            
 -   `Количество редакций после отправки`  - сколько раз заявка редактировалась после ее отправки;        
 -   `Количество отмен` - сколько раз заявка отменялась к моменту последней редакции заявки перед отправкой;       
 -   `Количество переносов` - сколько раз заявка переносилась к моменту последней редакции заявки перед отправкой;            
 -   `Среднее время между редакциями` - сколько дней проходило в среднем между редакциями. 
            
     

**Целевой признак (результат)**: статус выполнения заявки (1 -- если заявка выполнена, 0 -- если заявка не выполнена (перенесена, отменена)). 

In [4]:
df = pd.read_excel('orders_short_v3.xlsx')

# Исследовательский анализ

## Описательные статистики

In [5]:
obj = [
    'Идентификатор погрузки', 
    'Оператор', 
    'Учетное направление', 
    'Кластер назначения', 
    'Кластер отправления',
    'Код назначения', 
    'Код отправления', 
    'Идентификатор переноса', 
    'Результат']

df[obj].astype('object').describe(include = "O")

,Идентификатор погрузки,Оператор,Учетное направление,Кластер назначения,Кластер отправления,Код назначения,Код отправления,Идентификатор переноса,Результат
count,3190,3190,3190,3190,3190,3190,3190,3190,3190
unique,3190,31,12,24,21,149,144,3,2
top,6832,Rodkevicha,(8) ВРП,НСК,НСК,85160,85160,-1,1
freq,1,500,664,749,760,655,662,2003,1827


In [6]:
numerical = [
    'версия погрузки',
    'Количество вагонов',
    'Контрибуция', 
    'Длительность жизни',
    'Время до отправки',
    'Количество редакций после отправки',
    'Количество отмен', 
    'Количество переносов',
    'Среднее время между редакциями', 
    'Длительность афтершока']

df[numerical].describe()

,версия погрузки,Количество вагонов,Контрибуция,Длительность жизни,Время до отправки,Количество редакций после отправки,Количество отмен,Количество переносов,Среднее время между редакциями,Длительность афтершока
count,3190.000000,3190.000000,3.190000e+03,3190.000000,3190.000000,3190.000000,3190.000000,3190.000000,3190.000000,3190.000000
mean,3.759248,31.578056,1.062770e+06,9.139006,10.191777,0.863950,2.765517,0.657994,3.099542,-3.239984
std,2.608998,10.529060,2.476254e+06,10.854553,10.282935,1.501432,1.895896,1.172737,5.142586,14.826444
min,1.000000,0.000000,-6.522400e+07,0.000000,0.070000,0.000000,0.000000,0.000000,-1.000000,-44.480000
25%,2.000000,31.000000,0.000000e+00,0.000000,1.700000,0.000000,1.000000,0.000000,0.000000,-11.537500
50%,3.000000,31.000000,0.000000e+00,5.095000,5.795000,0.000000,2.000000,0.000000,1.200000,-0.750000
75%,5.000000,31.000000,1.974068e+06,15.282500,16.540000,1.000000,4.000000,1.000000,4.000000,3.277500
max,27.000000,71.000000,1.519000e+07,61.820000,53.040000,27.000000,17.000000,9.000000,42.000000,104.510000


## Проверка на нормальность

### Гистограмма

In [7]:
fig = go.Figure(data = [go.Histogram(x = df[i]) for i in numerical])
fig.show()

In [8]:
data = [list(df[i].values) for i in numerical]

fig = ff.create_distplot(
    data, numerical)

fig.show()

In [9]:
fig.for_each_trace(lambda t:  t.update(visible = 'legendonly') if not (t.name == 'Контрибуция') else t )

mask = np.eye(len(numerical)).astype(bool)

buttons = [dict(label = numerical[i], method = 'update', args = [{'visible': mask[i]}]) for i in range(len(numerical))]

fig.update_layout(
    updatemenus = list([
    dict(#active=1,
         buttons=buttons,
         direction = 'down',
         pad = {'r': 5, 't': 5},
         x=0.5,
         y=1.2
        
    )
]))

fig.show()

In [10]:
app = Dash(__name__)


app.layout = html.Div([
    html.H4('Интерактивная гистограмма'),
    html.P('Признак:'),
    dcc.Dropdown(numerical, 'Контрибуция', id='dropdown_value'),
    dcc.Graph(id="graph"),
      
])


@app.callback(
    Output("graph", "figure"), 
    Input("dropdown_value", "value"))
def display_hist(dropdown_value):
    fig = ff.create_distplot([list(df[dropdown_value].values)], [dropdown_value], show_rug=True)
    return fig

app.run(debug=True)

### Кумулятивное распределение

In [11]:
var = 'Длительность афтершока'

In [12]:
fig = px.ecdf(data_frame = df, x = [var])
fig.show()

In [13]:
xmin, xmax = df[var].min(), df[var].max()
mu, std = norm.fit(df[var])
x_normal = np.linspace(xmin, xmax, 100)
p = norm.cdf(x_normal, mu, std)

In [14]:
fig.add_trace(go.Scatter(x=x_normal, y=p, name = 'Нормальное распределение'))
fig.show()

In [15]:
app = Dash(__name__)


app.layout = html.Div([
    html.H4('Интерактивная гистограмма'),
    html.P('Признак:'),
    dcc.Dropdown(numerical, 'Контрибуция', id='dropdown_value'),
    dcc.Graph(id="graph"),
      
])


@app.callback(
    Output("graph", "figure"), 
    Input("dropdown_value", "value"))
def display_ecdf(dropdown_value):
    fig = px.ecdf(data_frame = df, x = [dropdown_value])
    xmin, xmax = df[dropdown_value].min(), df[dropdown_value].max()
    mu, std = norm.fit(df[dropdown_value])
    x = np.linspace(xmin, xmax, 100)
    p = norm.cdf(x, mu, std)
    fig.add_trace(go.Scatter(x=x, y=p, name = 'Нормальное распределение'))
    return fig

app.run(debug=True)


### График квантиль -  квантиль

**График квантиль-квантиль (график qq)** - это график рассеяния, на который наносятся значения набора данных в зависимости от значений нормального распределения для квантилей, определенных из набора данных. 
Координата y графика qq - это значения набора данных, координаты x - значения из нормального распределения.

In [16]:
var = ['Контрибуция']

In [17]:
perc_real = np.percentile(df[var], range(100))
#ds_n = StandardScaler().fit_transform(df[[dropdown_value]])
#perc_real = np.percentile(ds_n, range(100))

In [18]:
x_normal = np.random.normal(0,1, 1000)
perc_theoretical = np.percentile(x_normal, range(100))

In [19]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = perc_theoretical, y = perc_real, name = 'Actual fit_1', mode = 'markers'))
fig.add_trace(go.Scatter(x = perc_theoretical, y = perc_theoretical, name = 'Line of perfect fit' ))

### Проверка с использованием коэффициентов ассиметрии и эксцесса

In [20]:
from scipy.stats import skew, kurtosis

In [21]:
[print('skew = ',skew(df[column]), ', kurtosis = ', kurtosis(df[column])) for column in numerical];


skew =  1.748287860596319 , kurtosis =  5.282865667411514
skew =  0.3872118631388484 , kurtosis =  4.2947614320506595
skew =  -4.9196543968483395 , kurtosis =  163.58851484233497
skew =  1.133928239446145 , kurtosis =  0.45626062513430865
skew =  1.0774473015318526 , kurtosis =  0.23874277454087922
skew =  3.8117202661833924 , kurtosis =  35.247292805991044
skew =  1.3428666049948517 , kurtosis =  3.1670185066944105
skew =  2.500764088039889 , kurtosis =  7.894587711720888
skew =  3.249226974094131 , kurtosis =  13.754637800750565
skew =  0.24381967177278624 , kurtosis =  1.7775204106870248


### Проверка с использованием тестов

- тест Шапиро-Уилка – для небольших размеров выборки (<50 выборок)
- тест Колмогорова-Смирнова – для n ≥ 50

Для обоих вышеуказанных тестов нулевая гипотеза $H_0$ утверждает, что данные взяты из нормально распределенной совокупности.

In [22]:
from scipy.stats import shapiro, kstest 

In [23]:
[kstest(df[column], 'norm') for column in numerical]

[KstestResult(statistic=0.8413447460685429, pvalue=0.0, statistic_location=1, statistic_sign=-1),
 KstestResult(statistic=0.9801548041627273, pvalue=0.0, statistic_location=3, statistic_sign=-1),
 KstestResult(statistic=0.486833855799373, pvalue=0.0, statistic_location=1000, statistic_sign=-1),
 KstestResult(statistic=0.5591023718138619, pvalue=0.0, statistic_location=2.66, statistic_sign=-1),
 KstestResult(statistic=0.7316167185199995, pvalue=0.0, statistic_location=1.46, statistic_sign=-1),
 KstestResult(statistic=0.5, pvalue=0.0, statistic_location=0, statistic_sign=-1),
 KstestResult(statistic=0.7899340877613329, pvalue=0.0, statistic_location=1, statistic_sign=-1),
 KstestResult(statistic=0.5, pvalue=0.0, statistic_location=0, statistic_sign=-1),
 KstestResult(statistic=0.493730407523511, pvalue=0.0, statistic_location=0.0, statistic_sign=-1),
 KstestResult(statistic=0.4307226034215783, pvalue=0.0, statistic_location=-2.26, statistic_sign=1)]

**Вывод**: pvalue < 0.05 $\rightarrow$ есть основания отвергнуть гипотезу о нормальности распределения.

## Влияние количественных признаков на результат

Методы сравнения $Y$ качественный и $X$ количественный

<table>
<thead>
  <tr>
    <th colspan="2"></th>
    <th colspan="2"><center>Две выборки</center></th>
    <th colspan="2"><center>Больше двух выборок</center></th>
  </tr>
</thead>
    <thead>
  <tr>
    <th colspan="2"></th>
    <td><center>Независимые</center></td>
    <td><center>Зависимые</center></td>
    <td><center>Независимые</center></td>
    <td><center>Зависимые</center></td>
  </tr>
</thead>
<tbody>

  <tr>
    <td rowspan="2">Признак X</td>
    <td>Метрический<br>(параметрические методы сравнения)</td>
    <td>t-Стьюдента для независимых выборок</td>
    <td>t-Стьюдента для зависимых выборок</td>
    <td>ANOVA</td>
    <td>ANOVA с повторными измерениями</td>
  </tr>
  <tr>
    <td>Ранговый<br>(непараметрические методы сравнения)</td>
    <td>U - Манна-Уитни</td>
    <td>- T - Вилкоксона <br>- критерий знаков</td>
    <td>H - Краскала-Уоллеса</td>
    <td>$\chi^2$ Фридмана</td>
  </tr>
</tbody>
</table>

**Зависимые (парные) выборки** -- это когда получаем результат до и после "воздействия" для одного и того же набора респондентов. 
Например оценки студентов за контрольную точку до и после разбора заданий.

### Равенство средних для двух независимых выборок

$H_0$: средние равны.

1. Тестируем выборки на нормальность.
2. Если оба нормальные, то применяем Т-тест средних для двух независимых выборок.
3. Если хотя бы одно распределение отличается от нормального или объем выборки меньше 30 наблюдений, то применяем критерий Манна-Уитни.

In [24]:
from scipy.stats import mannwhitneyu, ttest_ind

In [25]:
df.groupby('Результат').agg({'Контрибуция':'mean'})

,Контрибуция
Результат,
0,1.063270e+06
1,1.062397e+06


In [26]:
column = 'Результат'
groups = [df[df[column] == i]['Контрибуция'].values for i in df[column].unique() ]
[kstest(i, 'norm') for i in groups]

[KstestResult(statistic=0.48789435069699194, pvalue=1.5728333796824728e-299, statistic_location=1000, statistic_sign=-1),
 KstestResult(statistic=0.4860426929392446, pvalue=0.0, statistic_location=23925, statistic_sign=-1)]

In [40]:
groups

[array([-1109524, -1109524,  -839714, ...,  3087790,  1958354,        0],
       dtype=int64),
 array([-394944, -259944, -843599, ...,       0, 1958354, 1460348],
       dtype=int64)]

In [27]:
mannwhitneyu(*groups)

MannwhitneyuResult(statistic=1294733.5, pvalue=0.04922244048461421)

**Задание:** 
1. Написать функцию, которая будет проверять, равны ли средние для двух выборок или нет. При выборе тесста учесть проверку выборок на нормальность распределения. 
2. Отобрать признаки, для которых наблюдаются достоверные различия в средних значениях.
3. Отобразить динамический модуль, в котором отображается гистограммы распределения заданного количественного признака для двух выборок: отправленных и неотправленных заявок. Количественные признаки перечислены в выпадающем списке.

In [58]:
def compare_means(sample_1, sample_2, alpha=0.05):
    normal = shapiro(sample_1).pvalue > alpha and shapiro(sample_2).pvalue > alpha

    if normal and min(len(sample_1), len(sample_2)) >= 30:
        result = ttest_ind(sample_1, sample_2)
        test = 't-критерий Стьюдента'
    else:
        result = mannwhitneyu(sample_1, sample_2)
        test = 'критерий Манна—Уитни'

    conclusion = 'средние равны' if result.pvalue >= alpha else 'средние различаются'
    print(f'{test}: p-value = {result.pvalue:.4f}. Вывод: {conclusion}')
    return result.pvalue >= alpha

In [57]:
l = [
    'Контрибуция',
    'Длительность жизни',
    'Время до отправки',
    'Среднее время между редакциями',
    'Длительность афтершока'
]

a = []

for j in l:
    samples = [df[df['Результат'] == i][j] for i in [0, 1]]
    if not compare_means(*samples):
        a.append(j)

print('Признаки со значимыми различиями:', a)

критерий Манна—Уитни: p-value = 0.0492. Вывод: средние различаются
критерий Манна—Уитни: p-value = 0.0624. Вывод: средние равны
критерий Манна—Уитни: p-value = 0.0000. Вывод: средние различаются
критерий Манна—Уитни: p-value = 0.5084. Вывод: средние равны
критерий Манна—Уитни: p-value = 0.0000. Вывод: средние различаются
Признаки со значимыми различиями: ['Контрибуция', 'Время до отправки', 'Длительность афтершока']


In [56]:
app_hist = Dash(__name__)

app_hist.layout = html.Div([
    dcc.Dropdown(l, l[0], id='feature'),
    dcc.Graph(id='histogram')
])

@app_hist.callback(Output('histogram', 'figure'), Input('feature', 'value'))
def histogram(feature):
    data = df[[feature, 'Результат']].copy()
    data['Статус'] = data['Результат'].map({1: 'Отправленные', 0: 'Неотправленные'})
    return px.histogram(data, x=feature, color='Статус', barmode='overlay')

app_hist.run(debug=True)